In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import DataLoader
from torchvision import transforms,datasets

In [2]:
transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,),(0.3081,))
])

In [3]:
trainset=datasets.MNIST(root='./data2',train=True,download=True,transform=transform)
testset=datasets.MNIST(root='./data2',train=False,download=True,transform=transform)

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 480kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.45MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.1MB/s]


In [4]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
testloader=DataLoader(testset,batch_size=64)

In [5]:
#Build the CNN
class CNN(nn.Module):
  def __init__(self):
    super(CNN,self).__init__()

    self.conv_layers=nn.Sequential(
        nn.Conv2d(1,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2))

    self.fc_layers=nn.Sequential(
        nn.Linear(128*3*3,256),
        nn.ReLU(),
        nn.Linear(256,10)
    )

  def forward(self,x):
    x=self.conv_layers(x)
    x=x.view(x.size(0),-1)
    x=self.fc_layers(x)
    return x







In [6]:
device=torch.device("cuda")
model=CNN().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [7]:
#Training the CNN
epochs=10
for epoch in range(epochs):
  epoch_training_loss=0.0
  for images,labels in trainloader:
    images,labels=images.to(device),labels.to(device)
    optimizer.zero_grad()
    output=model(images)
    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()
    epoch_training_loss+=loss.item()

  print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")






epoch=1/10 & loss=0.13702481601498714
epoch=2/10 & loss=0.04025907685449878
epoch=3/10 & loss=0.02779612915536976
epoch=4/10 & loss=0.023470308600537934
epoch=5/10 & loss=0.01832618964107475
epoch=6/10 & loss=0.013502922161349679
epoch=7/10 & loss=0.012316330862297942
epoch=8/10 & loss=0.010248670893919133
epoch=9/10 & loss=0.008975807511507285
epoch=10/10 & loss=0.008580523314420566


In [8]:
#Evaluate our CNN
correct_labels=0
total_labels=0
model.eval()
with torch.no_grad():
  for images,labels in testloader:
    images,labels=images.to(device),labels.to(device)
    outputs=model.forward(images)
    _,predicted=torch.max(outputs,1)

    correct_labels+=(predicted==labels).sum().item()
    total_labels+=labels.size(0)

print(f"accuracy={correct_labels/total_labels*100}")

accuracy=99.2


In [9]:
from sklearn.metrics import confusion_matrix
all_preds=predicted.cpu().numpy()
all_labels=labels.cpu().numpy()
cm=confusion_matrix(all_labels,all_preds)
print(cm)

[[1 0 0 0 0 0 0 0 0 0]
 [0 2 0 0 0 0 0 0 0 0]
 [0 0 2 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 2 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 0 0]
 [0 0 0 0 0 0 2 0 0 0]
 [0 0 0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 0 0 1]]


In [61]:
#Build our RNN
class RNN(nn.Module):
  def __init__(self):
    super().__init__()
    #self.hidden_size=hidden_size
    #self.num_layers=num_layers
    #RNN layers
    self.rnn=nn.LSTM(input_size=28,hidden_size=128,batch_first=True)
    #FC layers
    self.fc=nn.Linear(128,10)

  def forward(self,x):
    #h0=torch.zeros(self.num_layers,x.size(0),self.hidden_size)
    out,_=self.rnn(x)
    out=self.fc(out[:,-1,:])
    return out

device=torch.device('cuda')
model=RNN().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())

In [64]:
#training the RNN
epochs=10
for epoch in range(epochs):
  model.train()
  for xb,yb in trainloader:
    xb,yb=xb.to(device),yb.to(device)
    optimizer.zero_grad()
    xb=xb.squeeze()
    outputs=model(xb)
    #outputs=torch.sigmoid(outputs)
    loss=criterion(outputs,yb)
    loss.backward()
    optimizer.step()
  print(f"{epoch}/{epochs} and loss={loss.item()}")

0/10 and loss=0.029180962592363358
1/10 and loss=0.012688376009464264
2/10 and loss=0.005966124590486288
3/10 and loss=0.0052775489166378975
4/10 and loss=0.0002070464688586071
5/10 and loss=0.11746428906917572
6/10 and loss=0.01455385610461235
7/10 and loss=0.00629289960488677
8/10 and loss=0.0001776297576725483
9/10 and loss=0.0032580848783254623


In [65]:
#evaluate
correct_vals=0
tot_vals=0
model.eval()
with torch.no_grad():
  #correct_vals=0
  #tot_vals=0
  for xb,yb in testloader:
    xb,yb=xb.to(device),yb.to(device)
    xb=xb.squeeze(1) if xb.dim()==4 else xb

    xb=xb.squeeze()
    outputs=model(xb)
    _,predicted=torch.max(outputs,1)
    #predicted=predicted.squeeze()

    tot_vals+=yb.size(0)
    correct_vals+=(predicted==yb).sum().item()
print(f"accyracy={correct_vals/tot_vals*100}")



accyracy=98.6
